In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Environment ready")
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "PCB_DATASET"

IMAGE_ROOT = DATASET_ROOT / "Images"
ANNOTATION_ROOT = DATASET_ROOT / "Annotations"
TEMPLATE_ROOT = DATASET_ROOT / "PCB_USED"
ROTATION_ROOT = DATASET_ROOT / "rotation"

print(DATASET_ROOT.exists())
print(IMAGE_ROOT.exists())
print(ANNOTATION_ROOT.exists())
print(TEMPLATE_ROOT.exists())

In [ ]:
import cv2
import matplotlib.pyplot as plt

defective_path = IMAGE_ROOT / "Missing_hole" / "01_missing_hole_01.jpg"
reference_path = TEMPLATE_ROOT / "01.JPG"

defective = cv2.imread(str(defective_path))
reference = cv2.imread(str(reference_path))

print(defective.shape)
print(reference.shape)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(reference, cv2.COLOR_BGR2RGB))
plt.title("Reference PCB")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(defective, cv2.COLOR_BGR2RGB))
plt.title("Defective PCB")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
if reference.shape[:2] != defective.shape[:2]:
    raise ValueError(
        f"Image dimensions must match: {reference.shape[:2]} != {defective.shape[:2]}"
    )

reference_gray = cv2.cvtColor(reference, cv2.COLOR_BGR2GRAY)
defective_gray = cv2.cvtColor(defective, cv2.COLOR_BGR2GRAY)

# Absolute difference is an Otsu-specific input, not shared preprocessing.
difference = cv2.absdiff(reference_gray, defective_gray)

plt.figure(figsize=(8, 5))
plt.imshow(difference, cmap="gray")
plt.title("Reference–Test Difference")
plt.axis("off")
plt.show()

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path


def parse_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    objects = []

    for obj in root.findall("object"):
        box = obj.find("bndbox")

        objects.append({
            "class": obj.findtext("name"),
            "xmin": int(box.findtext("xmin")),
            "ymin": int(box.findtext("ymin")),
            "xmax": int(box.findtext("xmax")),
            "ymax": int(box.findtext("ymax")),
        })

    return objects

annotation_path = (
    ANNOTATION_ROOT
    / "Missing_hole"
    / "01_missing_hole_01.xml"
)

ground_truth_boxes = parse_annotation(annotation_path)

ground_truth_boxes

In [ ]:
annotated_defective = defective.copy()

for box in ground_truth_boxes:
    cv2.rectangle(
        annotated_defective,
        (box["xmin"], box["ymin"]),
        (box["xmax"], box["ymax"]),
        (0, 0, 255),
        4
    )

    cv2.putText(
        annotated_defective,
        box["class"],
        (box["xmin"], max(30, box["ymin"] - 10)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 0, 255),
        2
    )

plt.figure(figsize=(12, 6))
plt.imshow(cv2.cvtColor(annotated_defective, cv2.COLOR_BGR2RGB))
plt.title("Ground-Truth PCB Defects")
plt.axis("off")
plt.show()

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from algorithms.common import load_image, preprocess_pair

reference = load_image(reference_path)
defective = load_image(defective_path)

reference_processed, defective_processed = preprocess_pair(
    reference,
    defective
)
difference = cv2.absdiff(reference_processed, defective_processed)

print(reference_processed.shape)
print(defective_processed.shape)
print(difference.shape)